### Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

### Paths

In [2]:
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()

RAW_511_DIR = REPO_ROOT / "data" / "raw" / "511"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

DIM_PATH = INTERIM_DIR / "dim_neighbourhoods.parquet"
EVENTS_PATH = RAW_511_DIR / "events_snapshot.parquet"
CONSTRUCTION_PATH = RAW_511_DIR / "construction_snapshot.parquet"

print("DIM_PATH:", DIM_PATH)
print("EVENTS_PATH:", EVENTS_PATH)
print("CONSTRUCTION_PATH:", CONSTRUCTION_PATH)

DIM_PATH: C:\code\pyspark-playground\Covercheck-Toronto\data\interim\dim_neighbourhoods.parquet
EVENTS_PATH: C:\code\pyspark-playground\Covercheck-Toronto\data\raw\511\events_snapshot.parquet
CONSTRUCTION_PATH: C:\code\pyspark-playground\Covercheck-Toronto\data\raw\511\construction_snapshot.parquet


### Load data

In [3]:
dim = gpd.read_parquet(DIM_PATH)
events_df = pd.read_parquet(EVENTS_PATH)
construction_df = pd.read_parquet(CONSTRUCTION_PATH)

print("dim:", dim.shape)
print("events_df:", events_df.shape)
print("construction_df:", construction_df.shape)

dim: (158, 4)
events_df: (292, 29)
construction_df: (57, 26)


### Keep only needed neighbourhood columns

In [4]:
dim = dim[["nbhd_id", "area_name", "geometry"]].copy()
dim = gpd.GeoDataFrame(dim, geometry="geometry", crs="EPSG:4326")

print(dim.head())

   nbhd_id                  area_name  \
0      174  South Eglinton-Davisville   
1      173              North Toronto   
2      172         Dovercourt Village   
3      171   Junction-Wallace Emerson   
4      170         Yonge-Bay Corridor   

                                            geometry  
0  MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ...  
1  MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ...  
2  MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ...  
3  MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4...  
4  MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ...  


### Add datetime columns safely

In [5]:
def add_datetime_columns(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["Reported", "LastUpdated", "StartDate", "PlannedEndDate"]:
        if c in df.columns:
            df[c + "_dt"] = pd.to_datetime(df[c], unit="s", utc=True, errors="coerce")
    return df

events_df = add_datetime_columns(events_df)
construction_df = add_datetime_columns(construction_df)

events_df[[c for c in events_df.columns if c.endswith("_dt")]].head()

,Reported_dt,LastUpdated_dt,StartDate_dt,PlannedEndDate_dt
0,2025-09-10 11:00:00+00:00,2025-09-10 19:21:00+00:00,2025-09-10 11:00:00+00:00,2026-07-31 23:00:00+00:00
1,2025-09-10 11:00:00+00:00,2025-09-10 19:23:00+00:00,2025-09-10 11:00:00+00:00,2026-07-31 23:00:00+00:00
2,2025-09-10 11:00:00+00:00,2025-09-10 19:24:00+00:00,2025-09-10 11:00:00+00:00,2026-07-31 23:00:00+00:00
3,2025-10-07 15:30:00+00:00,2025-10-07 15:31:00+00:00,2025-10-07 15:30:00+00:00,2026-07-26 03:59:00+00:00
4,2025-10-09 11:00:00+00:00,2025-10-09 18:11:00+00:00,2025-10-09 11:00:00+00:00,2026-07-25 23:00:00+00:00


### Basic coordinate cleaning

In [6]:
def clean_coords(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

    df = df.dropna(subset=["Latitude", "Longitude"])

    # remove impossible values
    df = df[
        df["Latitude"].between(-90, 90) &
        df["Longitude"].between(-180, 180)
    ].copy()
    return df

events_df = clean_coords(events_df)
construction_df = clean_coords(construction_df)

print("events after coord clean:", events_df.shape)
print("construction after coord clean:", construction_df.shape)

events after coord clean: (292, 29)
construction after coord clean: (57, 26)


### Convert to GeoDataFrames

In [7]:
events_gdf = gpd.GeoDataFrame(
    events_df,
    geometry=gpd.points_from_xy(events_df["Longitude"], events_df["Latitude"]),
    crs="EPSG:4326"
)

construction_gdf = gpd.GeoDataFrame(
    construction_df,
    geometry=gpd.points_from_xy(construction_df["Longitude"], construction_df["Latitude"]),
    crs="EPSG:4326"
)

print(events_gdf.shape, construction_gdf.shape)

(292, 30) (57, 27)


### Spatial filter to Toronto neighbourhoods

In [8]:
events_tor = gpd.sjoin(
    events_gdf,
    dim[["nbhd_id", "area_name", "geometry"]],
    how="inner",
    predicate="within"
)

construction_tor = gpd.sjoin(
    construction_gdf,
    dim[["nbhd_id", "area_name", "geometry"]],
    how="inner",
    predicate="within"
)

print("events_tor:", events_tor.shape)
print("construction_tor:", construction_tor.shape)

print("Toronto neighbourhoods hit by events:", events_tor["nbhd_id"].nunique() if len(events_tor) else 0)
print("Toronto neighbourhoods hit by construction:", construction_tor["nbhd_id"].nunique() if len(construction_tor) else 0)

events_tor: (31, 33)
construction_tor: (2, 30)
Toronto neighbourhoods hit by events: 13
Toronto neighbourhoods hit by construction: 2


### Build severity weights

In [9]:
SEVERITY_MAP = {
    "minor": 1,
    "moderate": 2,
    "major": 3,
    "unknown": 1
}

def map_severity(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    return s.map(SEVERITY_MAP).fillna(1).astype(int)

if "Severity" in events_tor.columns:
    events_tor["severity_weight"] = map_severity(events_tor["Severity"])
else:
    events_tor["severity_weight"] = 1

# Construction feed doesn't appear to have a Severity field
construction_tor["severity_weight"] = 1

### Create full closure flags

In [10]:
def full_closure_flag(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1", "yes"]).astype(int)

if "IsFullClosure" in events_tor.columns:
    events_tor["full_closure_flag"] = full_closure_flag(events_tor["IsFullClosure"])
else:
    events_tor["full_closure_flag"] = 0

if "IsFullClosure" in construction_tor.columns:
    construction_tor["full_closure_flag"] = full_closure_flag(construction_tor["IsFullClosure"])
else:
    construction_tor["full_closure_flag"] = 0

### Expand each record across active dates

In [11]:
def expand_active_days(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()

    # start date
    df["start_date"] = pd.to_datetime(df["StartDate_dt"], utc=True, errors="coerce").dt.date

    # end date: if missing, fall back to start date
    df["end_date"] = pd.to_datetime(df["PlannedEndDate_dt"], utc=True, errors="coerce").dt.date
    df["end_date"] = df["end_date"].fillna(df["start_date"])

    # handle bad end < start
    bad_mask = df["end_date"] < df["start_date"]
    df.loc[bad_mask, "end_date"] = df.loc[bad_mask, "start_date"]

    # expand each row into list of active dates
    df["active_dates"] = df.apply(
        lambda r: pd.date_range(r["start_date"], r["end_date"], freq="D").date
        if pd.notna(r["start_date"]) and pd.notna(r["end_date"]) else [],
        axis=1
    )

    expanded = df.explode("active_dates").copy()
    expanded = expanded.rename(columns={"active_dates": "date"})
    expanded["source_type"] = source_name

    return expanded

events_expanded = expand_active_days(events_tor, "event")
construction_expanded = expand_active_days(construction_tor, "construction")

print("events_expanded:", events_expanded.shape)
print("construction_expanded:", construction_expanded.shape)

events_expanded: (124, 39)
construction_expanded: (2325, 36)


### Build daily neighbourhood aggregates

In [12]:
# Events daily
events_daily = (
    events_expanded.groupby(["date", "nbhd_id"], as_index=False)
    .agg(
        events_count=("ID", "count"),
        events_severity_weighted=("severity_weight", "sum"),
        events_full_closure_count=("full_closure_flag", "sum")
    )
)

# Construction daily
construction_daily = (
    construction_expanded.groupby(["date", "nbhd_id"], as_index=False)
    .agg(
        construction_count=("ID", "count"),
        construction_severity_weighted=("severity_weight", "sum"),
        construction_full_closure_count=("full_closure_flag", "sum")
    )
)

print("events_daily:", events_daily.shape)
print("construction_daily:", construction_daily.shape)

events_daily: (85, 5)
construction_daily: (2325, 5)


### Merge both into a single 511 daily table

In [13]:
ont511_daily = pd.merge(
    events_daily,
    construction_daily,
    on=["date", "nbhd_id"],
    how="outer"
)

count_cols = [
    "events_count",
    "events_severity_weighted",
    "events_full_closure_count",
    "construction_count",
    "construction_severity_weighted",
    "construction_full_closure_count"
]

for c in count_cols:
    if c in ont511_daily.columns:
        ont511_daily[c] = ont511_daily[c].fillna(0).astype(int)

ont511_daily = ont511_daily.sort_values(["date", "nbhd_id"]).reset_index(drop=True)

print("ont511_daily shape:", ont511_daily.shape)
ont511_daily.head()

ont511_daily shape: (2410, 8)


,date,nbhd_id,events_count,events_severity_weighted,events_full_closure_count,construction_count,construction_severity_weighted,construction_full_closure_count
0,2022-04-04,150,0,0,0,1,1,0
1,2022-04-05,150,0,0,0,1,1,0
2,2022-04-06,150,0,0,0,1,1,0
3,2022-04-07,150,0,0,0,1,1,0
4,2022-04-08,150,0,0,0,1,1,0


### sanity checks

In [14]:
print("Date range:", ont511_daily["date"].min(), "to", ont511_daily["date"].max())
print("Neighbourhoods covered:", ont511_daily["nbhd_id"].nunique())
print(ont511_daily.describe(include="all"))

Date range: 2022-04-04 to 2026-04-02
Neighbourhoods covered: 13
              date    nbhd_id  events_count  events_severity_weighted  \
count         2410     2410.0   2410.000000               2410.000000   
unique        1380       <NA>           NaN                       NaN   
top     2026-03-12       <NA>           NaN                       NaN   
freq             9       <NA>           NaN                       NaN   
mean           NaN  85.065975      0.051452                  0.051452   
std            NaN  73.238321      0.314068                  0.314068   
min            NaN        1.0      0.000000                  0.000000   
25%            NaN        1.0      0.000000                  0.000000   
50%            NaN      150.0      0.000000                  0.000000   
75%            NaN      150.0      0.000000                  0.000000   
max            NaN      156.0      4.000000                  4.000000   

        events_full_closure_count  construction_count  \
co

### save

In [15]:
out_path = INTERIM_DIR / "ontario511_nbhd_daily.parquet"
ont511_daily.to_parquet(out_path, index=False)

print("Saved:", out_path)

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\interim\ontario511_nbhd_daily.parquet
